In [1]:
import json
import os

In [2]:
# load search_response_old.json
with open('search_response_old.json', 'r', encoding='utf-8') as f:
    old_data = json.load(f)


all offers list

In [3]:
# all offers list
all_offers = old_data.get('CatalogProductOfferingsResponse', {}).get('CatalogProductOfferings', {}).get('CatalogProductOffering', [])

unique values of "sequence"

In [4]:
# unique values of sequence
sequences = [all_offers[i]["sequence"] for i in range(len(all_offers))]
unique_sequences = list(set(sequences))

In [5]:
print("Unique sequences found:", unique_sequences)

Unique sequences found: [1]


How many offers are there? 

In [6]:
print(f"total offers found: {len(all_offers)}")

total offers found: 9


In [7]:
straightened_offers = []

- so each Leg response has multiple offers. Each having one flight  (7 in this case)
- each offer has multiple products (flights (timings) ) coupled with multiple brands(cabin class etc. )
- Price is dependent on products + brands conbination. SO each Product(flight sequence) will have a range of prices depending upon brand i.e. cabin class + amenities during that.
- So, for each user request, we should show him: 
    - all Products of all offers, and in each Product: 
        - Departure airport of Segment 1
        - Arrival Airport of  Segment 1
        - Departure Time of Segment 1
        - Arrival Time of Segment 1
        - Carrier of Segment 1
        - Available Cabin Classes of Segment 1
        - Price Range from minimum Brand offer to maximum Brand offer for Segment 1
        - Departure airport of Segment 2
        - Arrival Airport of  Segment 2
        - Departure Time of Segment 2
        - Arrival Time of Segment 2
        - Carrier of Segment 2
        - Available Cabin Classes of Segment 2
        - Price Range from minimum Brand offer to maximum Brand offer for Segment 2
        - And So on for each segment

In [8]:

s8 = {
            "@type": "FlightDetail",
            "distance": 3409,
            "duration": "PT6H55M",
            "carrier": "EK",
            "number": "10",
            "equipment": "388",
            "id": "s8",
            "Departure": {
              "@type": "DepartureDetail",
              "terminal": "N",
              "location": "LGW",
              "date": "2025-12-20",
              "time": "20:25:00"
            },
            "Arrival": {
              "@type": "ArrivalDetail",
              "terminal": "3",
              "location": "DXB",
              "date": "2025-12-21",
              "time": "07:20:00"
            },
            "AvailabilitySourceCode": "Q"
          },

In [9]:

s12 = {
            "@type": "FlightDetail",
            "distance": 3409,
            "duration": "PT7H5M",
            "carrier": "EK",
            "number": "16",
            "equipment": "388",
            "id": "s12",
            "Departure": {
              "@type": "DepartureDetail",
              "terminal": "N",
              "location": "LGW",
              "date": "2025-12-20",
              "time": "13:35:00"
            },
            "Arrival": {
              "@type": "ArrivalDetail",
              "terminal": "3",
              "location": "DXB",
              "date": "2025-12-21",
              "time": "00:40:00"
            },
            "AvailabilitySourceCode": "Q"
          },

In [10]:
refList = old_data.get("ReferenceList", [])

In [ ]:
# [RefList["Flight"]["number"] for id in offer["ProductBrandOptions"]["flightRefs"]
flag = False
offer = {}
ids = []
for id in offer["ProductBrandOptions"]["flightRefs"]:
    for flight in refList["Flight"]:
        if id == flight["number"]:
            ids.append(id)
        else:
            continue

# the list comprehension equivalent is: 

ids = [
    flight_id
    for flight_id in offer["ProductBrandOptions"]["flightRefs"]
    if flight_id in {f["number"] for f in refList["Flight"]}
]



In [12]:
for i, offer in enumerate(all_offers):
    straightened_offers.append(
        f"Offer {i}: {offer.get('id', [])} - Departure: {offer.get('Departure', [])} - Arrival: {offer.get('Arrival', [])} - How many brands: {len(offer.get("Brand", []))} - How many Products: {len(offer.get("ProductBrandOptions", []))} - How many Flights Segments per product: {len(offer["ProductBrandOptions"]["flightRefs"])} - Minimum Price: {min([productbrand["BestCombinablePrice"]["TotalPrice"] for productbrand in offer["ProductBrandOptions"]["ProductBrandOffering"]])} - Maximum Price: {max([productbrand["BestCombinablePrice"]["TotalPrice"] for productbrand in offer["ProductBrandOptions"]["ProductBrandOffering"]])} - Direct Flight? {("Yes" if len(offer["ProductBrandOptions"]["flightRefs"]) > 1 and len(set([flight_id for flight_id in offer["ProductBrandOptions"]["flightRefs"] if flight_id in {f["number"] for f in refList["Flight"]}])) else "No" )} "
    )



TypeError: list indices must be integers or slices, not str